<a href="https://colab.research.google.com/github/kanakamvasundhara/SCT_WD_1/blob/main/Research%20Paper%20Question%20Answering%20System" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pypdf
!pip install faiss-cpu
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 25.4 MB/s eta 0:00:00


In [7]:
pdf_file = "/content/nlp_application.pdf"

In [8]:
reader = PdfReader(pdf_file)

pages = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        pages.append(text)

print("Number of pages:", len(pages))
print("Text extracted successfully")

Number of pages: 15
Text extracted successfully


In [9]:
chunks = []
chunk_size = 500
overlap = 100

for page_no, text in enumerate(pages, start=1):
    words = text.split()

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])

        if len(chunk.strip()) > 50:
            chunks.append({
                "text": chunk,
                "page": page_no
            })

print("Total chunks:", len(chunks))

Total chunks: 24


In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (24, 384)


In [11]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print("Vectors stored:", index.ntotal)

Vectors stored: 24


In [12]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=250
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2

In [13]:
def ask_question(question, k=3):

    query_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    retrieved = []

    for idx in indices[0]:
        retrieved.append(chunks[idx])

    context = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    prompt = f"""
Answer the question using ONLY the information given in the context.

If the answer is not available in the context, say:
"Information not found in the research paper."

Give a short and clear answer.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generator(prompt)[0]["generated_text"]

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    for item in retrieved:
        print(f"- Page {item['page']}")

    return answer

In [ ]:
question = input("Enter your question: ")

ask_question(question)